# SAM3 auto-labeling — train/val pseudo-labels + zero-shot test benchmark

Runs on Colab (GPU required) — see `CLAUDE.md` Section 3. All logic lives in `src/`; this notebook only orchestrates calls, per Section 9 ("no notebook-only logic").

Roles (see `docs/decision_log.md`, 2026-09-12 entries):
- **Labeling tool**: pseudo-labels the `train` + `val` videos only. Never touches `test` — enforced in code by `scripts/04_label_video.py` (`check_not_test_video`), not just by convention.
- **Zero-shot detector (Tier 1, item 3)**: the *same* model, benchmarked on `test` in a separate notebook/script once the gold labels exist — not run from here.

**Before running:** request+download of `sam3.pt` needs your approved Hugging Face access (`facebook/sam3`) — see the HF-login cell below.

## 1. Clone the repo and install dependencies

Code lives on GitHub (`Kametor/object-detection-drone`, public) since it changes daily; the raw video data lives on Drive (mounted next) since it's large and gitignored. No authentication needed to clone — only step 8 (pushing results back) needs your token, since writing always requires credentials even on a public repo.

In [ ]:
!git clone https://github.com/Kametor/object-detection-drone.git
%cd object-detection-drone
!pip install -q -r requirements.txt

## 2. Mount Drive and link the raw video data

`DRIVE_VIDEO_DIR` is set to `/content/drive/MyDrive/object-detection/drone_videos` (your uploaded folder). If Drive mounts somewhere unexpected, adjust the path below to match — the `ls` at the end should list the 13 video files.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

DRIVE_VIDEO_DIR = '/content/drive/MyDrive/object-detection/drone_videos'

os.makedirs('data/raw', exist_ok=True)
if not os.path.islink('data/raw/drone_videos'):
    os.symlink(DRIVE_VIDEO_DIR, 'data/raw/drone_videos')

!ls "data/raw/drone_videos" | head

## 3. Hugging Face login and SAM3 weights

Requires your approved access to `facebook/sam3`. `login()` will prompt for an HF token (create one at huggingface.co/settings/tokens if needed).

The exact weight filename isn't 100% confirmed — `list_repo_files` prints what's actually there so we download the right one instead of guessing.

In [ ]:
from huggingface_hub import login, list_repo_files, hf_hub_download

login()  # paste your HF token when prompted

files = list_repo_files("facebook/sam3")
print(files)  # confirm the actual .pt filename before downloading

weight_filename = "sam3.pt"  # adjust if the printed list above shows a different name
sam3_checkpoint_path = hf_hub_download(repo_id="facebook/sam3", filename=weight_filename)
print("Downloaded to:", sam3_checkpoint_path)

## 4. Point the config at the downloaded checkpoint

Switches `configs/04_label_video.yaml` from the local `mock` default to real `sam3`, on GPU.

In [ ]:
import yaml

config_path = "configs/04_label_video.yaml"
config = yaml.safe_load(open(config_path))
config["model"]["name"] = "sam3"
config["model"]["checkpoint"] = sam3_checkpoint_path
config["model"]["device"] = "cuda"
yaml.safe_dump(config, open(config_path, "w"), sort_keys=False)

print(open(config_path).read())

## 5. Smoke test on one real frame

`Sam3Detector` hasn't been exercised against real weights yet (see `docs/decision_log.md`, 2026-09-12) — the exact `set_image`/`predictor(text=...)` call shape may need a small fix here before it's trusted for the full batch below. Don't skip this cell.

In [ ]:
import sys
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import yaml

sys.path.insert(0, ".")
from src.data.sampling import read_frame
from src.methods.sam3_zeroshot.model import Sam3Detector
from src.methods.sam3_zeroshot.pipeline import draw_detections

# Read the same config cell 6 will use, so this smoke test actually
# previews production behavior (including confidence_threshold) instead
# of using the detector's own unfiltered defaults.
config = yaml.safe_load(open("configs/04_label_video.yaml"))
detector = Sam3Detector(
    checkpoint=sam3_checkpoint_path, device="cuda", conf=config["confidence_threshold"]
)

test_video = Path("data/raw/drone_videos/Berghouse Leopard Jog.mp4")
frame = read_frame(test_video, frame_idx=0)
raw_detections = detector.detect(frame, "person")
detections = [d for d in raw_detections if d.score >= config["confidence_threshold"]]
print(f"{len(detections)} detections (threshold={config['confidence_threshold']}):", detections)

annotated = draw_detections(frame, detections)
plt.figure(figsize=(10, 6))
plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()

## 6. If the smoke test looks right: label all train + val videos

Runs `scripts/04_label_video.py` once per video — the same code path that would run for a reviewer's own uploaded video. `test.txt` videos are structurally refused (see `check_not_test_video` in the script).

In [ ]:
import subprocess
from pathlib import Path

for split in ["train", "val"]:
    videos = [v.strip() for v in Path(f"data/splits/{split}.txt").read_text().splitlines() if v.strip()]
    for video in videos:
        video_path = f"data/raw/drone_videos/{video}"
        print(f"--- [{split}] labeling {video} ---")
        subprocess.run(
            [
                "python", "scripts/04_label_video.py",
                "--video", video_path,
                "--prompt", "person",
                "--fps", "3.0",
                "--config", "configs/04_label_video.yaml",
            ],
            check=True,
        )

## 7. Spot-check a sample of review images

Boxes + confidence drawn on top of real frames — this is the human spot-check pass mentioned in `docs/decision_log.md` (2026-09-12, "SAM3's role"), on top of the automated quality filters that come next (Day 4, ByteTrack + NMS + noise audit).

In [ ]:
import random
from pathlib import Path

import cv2
import matplotlib.pyplot as plt

review_images = list(Path("results/pseudo_labels/review").rglob("*.jpg"))
sample = random.sample(review_images, min(9, len(review_images)))

fig, axes = plt.subplots(3, 3, figsize=(15, 10))
for ax, img_path in zip(axes.flat, sample):
    img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    ax.imshow(img)
    ax.set_title(img_path.parent.name + "/" + img_path.name, fontsize=8)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 8. Push the annotations back to GitHub

Only `results/pseudo_labels/annotations/*.json` and the run manifests are committed — `frames/` and `review/` are gitignored (bulky, regenerable by re-running this notebook) so the repo stays lightweight.

Running the next cell will prompt for your GitHub personal access token (input is masked, and it's never saved to disk).

In [ ]:
import getpass

# Pasted here, it's masked on screen and lives only in this running
# session's memory — never written to the notebook file or git history.
gh_token = getpass.getpass("GitHub personal access token: ")

!git config user.email "you@example.com"
!git config user.name "Colab"
!git add results/pseudo_labels/annotations results/manifests
!git commit -m "Add SAM3 pseudo-labels for train+val (Colab run)"
!git push https://{gh_token}@github.com/Kametor/object-detection-drone.git main